# Run Simulations 

Models 1, 2, 3

Runs SSA trajectories for each model across a range of SOS concentrations
and saves results to a `.pkl` file for use in the plotting notebooks.

**Models:**
- **Model 1** — baseline Lee et al. parameters, processive SOS, `[SOS] = 1–10 nM`
- **Model 2** — 10× higher kcat2, higher SOS, `[SOS] = 1–30 nM` → multimodal
- **Model 3** — non-processive (kon/koff × 1000), `[SOS] = 1–10 nM` → unimodal

Each trajectory is stored as a dict keyed by observable name.


In [6]:
import bionetgen
import numpy as np
import pickle


## Set model configuration

In [7]:

MODEL = "Model2"   #  "Model1" "Model2" "Model3"

# SOS concentrations (nM)
SOS_CONC = {
    "Model1": [1, 5, 10],
    "Model2": [1, 15, 30],
    "Model3": [1, 5, 10],
}[MODEL]

# Kon 
KON = {
    "Model1": 7e-8,   # processive bimodal
    "Model2": 7e-8,   # processive multimodal 
    "Model3": 7e-5,   # non-processive (1000× faster on/off)
}[MODEL]

T_END   = 100000   # simulation duration (s)
N_STEPS = 10000   # number of output time points

OBSERVABLES = ['totalRasGTP', 'totalBoundRas', 'TotalRas']


## Load and configure model

In [8]:
model = bionetgen.bngmodel(MODEL + ".bngl")
sim   = model.setup_simulator()
sim.setIntegrator('gillespie')
sim.selections = ['time'] + OBSERVABLES


## Run trajectories across SOS concentrations

In [9]:
trajectories = []

for sos in SOS_CONC:
    sim.RhoSOS = sos
    sim.Kon1   = sos * KON
    sim.Kon2   = sos * KON
    sim.reset()

    print(f"[SOS] = {sos} nM ...")
    result = sim.simulate(0, T_END, N_STEPS)

    col_names = ['time'] + OBSERVABLES
    traj = {name: result[:, i] for i, name in enumerate(col_names)}
    trajectories.append(traj)

print(f"Done. Each trajectory: {result.shape[0]} time points, t = 0 to {T_END} s")


[SOS] = 1 nM ...
[SOS] = 15 nM ...
[SOS] = 30 nM ...
Done. Each trajectory: 10000 time points, t = 0 to 100000 s


## Save to .pkl

In [10]:
out = {'model': MODEL, 'pval': SOS_CONC, 'traj': trajectories}

with open(MODEL + '_traj.pkl', 'wb') as f:
    pickle.dump(out, f)

print(f"Saved {MODEL}_traj.pkl")



Saved Model2_traj.pkl
